# Feature engineering — capstone narrative

Mirrors **`src/feature_engineering.py`** with **rationale**, **validation plots**, and **interview notes**.

**Derived fields:** `price_per_sqft`, `property_age`, `bed_bath_ratio`, `location_target_enc`, `location_cluster` (KMeans on lat/lon), regex-based amenity flags, `amenity_count`, `luxury_score`.


In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

clean = pd.read_csv(PROJECT_ROOT / "data/processed/texas_houston_clean.csv", low_memory=False)
print("Clean shape:", clean.shape)
clean.head(3)


## 1. Manual preview — price per sqft

Stabilizes comparability across different home sizes before tree/linear models.


In [ ]:
tmp = clean.copy()
tmp["price_per_sqft_preview"] = tmp["price"] / tmp["sqft"].replace(0, np.nan)
fig = px.histogram(tmp.dropna(subset=["price_per_sqft_preview"]), x="price_per_sqft_preview", nbins=80, title="price / sqft (preview)")
fig.update_layout(template="plotly_white")
fig.show()


## 2. Run production pipeline

`engineer_features()` writes `data/processed/texas_houston_features.csv`.


In [ ]:
from src.feature_engineering import engineer_features

out = engineer_features()
print("Wrote:", out)
feat = pd.read_csv(PROJECT_ROOT / "data/processed/texas_houston_features.csv")
new_cols = sorted(set(feat.columns) - set(clean.columns))
print("New columns:", new_cols)


## 3. Validate engineered distributions


In [ ]:
for col in ["price_per_sqft", "property_age", "luxury_score", "location_cluster"]:
    if col in feat.columns:
        display(feat[col].describe(percentiles=[0.05, 0.5, 0.95]).to_frame(col))


In [ ]:
sub = feat.sample(min(4000, len(feat)), random_state=2)
fig = px.scatter(sub, x="price_per_sqft", y="price", color="location_cluster", opacity=0.4, title="Price vs price_per_sqft (by geo cluster)")
fig.update_layout(template="plotly_white")
fig.show()

if "luxury_score" in feat.columns:
    fig2 = px.scatter(sub, x="luxury_score", y="price", trendline="ols", title="Price vs luxury_score")
    fig2.update_layout(template="plotly_white")
    fig2.show()


## 4. Multicollinearity spot-check

`sqft`, `price_per_sqft`, and `price` are mechanically related — tree ensembles handle this well; linear models may need regularization.


In [ ]:
num = [c for c in ["price", "sqft", "price_per_sqft", "property_age", "bed_bath_ratio", "luxury_score", "amenity_count"] if c in feat.columns]
cm = feat[num].corr()
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt=".2f", cmap="RdBu_r", center=0)
plt.title("Correlation: engineered + core numerics")
plt.tight_layout()
plt.show()


## 5. Interview notes

- **Target encoding:** use fold-wise or smoothed encoding in production to avoid leakage.
- **Clusters:** interpret `location_cluster` as a non-linear location proxy.
- **Text:** regex amenities are a baseline; NLP embeddings are a natural upgrade.
